# Apartado 6. Detección de contenido inapropiado

# Extracción de 10 hilos polémicos con 50 de sus comentarios

In [7]:
import json
from pathlib import Path
from datetime import datetime
from funciones_auxiliares import * 

# Parámetros solicitados en el enunciado 
HILOS_POLEMICOS = 10
COMENTARIOS_POR_HILO_COMMENT = 50
SUBREDDIT = "OpinionesPolemicas"

todas_submissions = []

# Extraer los 10 hilos (submissions)
# Filtramos hilos que tengan al menos 50 comentarios para asegurar el corpus
todas_submissions = extract_submissions(
	"datos/RS_OpinionesPolemicas_2025.zst", 
	[SUBREDDIT],
	n_submissions=HILOS_POLEMICOS, 
	min_comments=COMENTARIOS_POR_HILO_COMMENT
)

# 2. Extraer todos los comentarios de golpe
extract_comments_for_submissions(
	"datos/RC_OpinionesPolemicas_2025.zst", 
	todas_submissions, 
	num_comments=COMENTARIOS_POR_HILO_COMMENT
)

# 3 y 4. Separar la "superlista" y guardar en JSONs individuales
print("Generando archivos JSON individuales...")

# Filtramos la lista global para quedarnos solo con los hilos de ESTE subreddit
submissions_del_sub = [s for s in todas_submissions if s.get('subreddit', '').lower() == SUBREDDIT.lower()]

# Estructurar el resultado
resultado_final = {
		"subreddit": SUBREDDIT,
		"extraction_date": datetime.now().isoformat(),
		"num_submissions": len(submissions_del_sub),
		"total_comments": sum(len(s['comments']) for s in submissions_del_sub),
		"submissions": submissions_del_sub
	}

# Guardar en JSON individual
nombre_archivo = f"ejemplo_subreddit_{SUBREDDIT}.json"
with open(nombre_archivo, 'w', encoding='utf-8') as f:
	json.dump(resultado_final, f, ensure_ascii=False, indent=2)

print(f"✅ Archivo '{nombre_archivo}' generado con éxito.")


Post guardado en r/opinionespolemicas (1/10)
Post guardado en r/opinionespolemicas (2/10)
Post guardado en r/opinionespolemicas (3/10)
Post guardado en r/opinionespolemicas (4/10)
Post guardado en r/opinionespolemicas (5/10)
Post guardado en r/opinionespolemicas (6/10)
Post guardado en r/opinionespolemicas (7/10)
Post guardado en r/opinionespolemicas (8/10)
Post guardado en r/opinionespolemicas (9/10)
Post guardado en r/opinionespolemicas (10/10)
✅ ¡Todas las submissions encontradas!
🔍 Buscando ~50 comentarios por hilo...
✅ ¡Completado! No hace falta leer más el archivo.
✨ Extracción de comentarios finalizada.
Generando archivos JSON individuales...
✅ Archivo 'ejemplo_subreddit_OpinionesPolemicas.json' generado con éxito.


# Zero Shot Learning

In [ ]:
# Ver cuanto se relaciona nuestra premisa con nuestras etiquetas
tasks = [
    {
        'title': 'clasificación de opiniones',
        'text': 'El servicio en este restaurante fue increíble, definitivamente volveré.', # Estas son nuestras premisas
        'labels': ['positivo', 'negativo', 'neutral'] # Estas son nuestras etiquetas o hipótesis
    }, {
        'title': 'análisis de emociones',
        'text': 'No puedo creer que me haya pasado esto. Todo es un desastre',
        'labels': ['alegría', 'disgusto', 'ira', 'miedo', 'otro', 'sorpresa', 'tristeza']
    }, {
        'title': 'detección de intención',
        'text': 'Quiero devolver un producto porque llegó dañado',
        'labels': ['elogio', 'pregunta', 'queja', 'solicitud']
    },{
        'title': 'análisis de emociones',
        'text': 'No me ha gustado nada la atención recibida por los camareros.',
        'labels': ['alegría', 'disgusto', 'ira', 'miedo', 'otro', 'sorpresa', 'tristeza']
    }, {
        'title': 'clasificación de opiniones',
        'text': 'Qué suerte tengo, me ha tocado la loteria contigo.',
        'labels': ['positivo', 'negativo', 'neutral']
    }
]

# Más ejemplos
# No me ha gustado nada la atención recibida por los camareros.
# Qué suerte tengo, me ha tocado la loteria contigo.
# ¿Es normal que mi frigorífico me de la corriente cuando lo intento abrir?

In [ ]:
# Importamos las librerias necesarias
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Definimos el modelo que vamos a usar
model_path = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'

# Cargamos el tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Cargamos el modelo, recordad que este modelo ha sido preentenado para la tarea de NLI
# Según el tipo de tarea hay que cargar el modelo de formas diferentes
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Esta variable almacenará la respuesta del modelo
response = []

# Para cada tarea definida
for task in tasks:

  # Transformamos los textos en formato de premisa e hipótesis
  premises = [task['text']] * len(task['labels'])
  hypotheses = [f"El texto trata sobre {label}." for label in task['labels']]

  # Tokenizamos tanto las premisas como las hipótesis
  inputs = tokenizer(premises, hypotheses, return_tensors = "pt", padding = True, truncation = True)
  # padding = True, truncation = True: Para que todos tengan la misma longitud. Padding mejora el rendimiento, rellenar el texto con menor longitud hasta llegar a la longitud máxima de los demás, es opcional. Truncation, trunca.

  # Pasamos las premisas e hipótesis tokenizadas al modelo
  outputs = model(**inputs)

  # Obtenemos los logits del modelo
  # Estos son los valores de la última capa de nuestro modelo, es decir, las probabilidades de entailment, neutral y contradiction
  logits = outputs.logits
  # Sin embargo, esto son valores en crudo
  # Vamos a usar softmax para normalizarlos, haciendo que su suma sea 1 (probabilidades excluyentes)
  probs = torch.softmax(logits, dim = 1)

  # Al estar trabajando en NLI, la salida del modelo incluye varias columnas, como entailment, neutral y contradiction
  # Nos interesa quedarnos con la columna que contenga la clase "entailment", es decir, la columna que indica
  # la probabilidad de que la premisa (texto) implique la hipotesis (cada etiqueta)
  entailment_probs = probs[:, 0]
  # No todos los modelos lo almacenan en la misma columna, así que esto puede cambiar de modelo a modelo.
  # Para saber que columna es la correcta, puedes revisar la documentación del modelo en Hugging Face

  # Vamos a realizar un paso adicional para normalizar las probabilidades
  # Este paso es necesario para que la suma del entailment de todas las etiquetas sea 1
  normalized_probs = entailment_probs / entailment_probs.sum()

  # Asociamos cada etiqueta con su probabilidad
  label_probs = {label: prob.item() for label, prob in zip(task['labels'], normalized_probs)}

  # Ordenamos la salida de mayor a menor según la probabilidad
  label_probs = dict(sorted(label_probs.items(), key = lambda item: item[1], reverse = True))

  # Guardamos la respuesta obtenida
  response.append({
     'task': task['title'],
     'text': task['text'],
     'scores': label_probs
  })

# Mostramos la respuesta del modelo
print(json.dumps(response, indent = 4, sort_keys = False, ensure_ascii = False))

## Few Shot Learning

In [ ]:
# Definimos las etiquetas a clasificar
sentiment_labels = ["positivo", "negativo", "neutro"]

# Textos a clasificar
sentiment_sentences = [
  "Esta película me pareció maravillosa",
  "La película me pareció muy mala",
  "Viendo esa película me aburrí como una ostra"
]

# Definimos un par de ejemplos para incluirlos en el prompt
# Los ejemplos deben estar relacionados con la tarea a resolver
few_shot_examples = """
Ejemplo 1: Desde la semana pasada no me lo pasaba tan bien -> positivo
Ejemplo 2: A tí la película te gustó, pero a mí no -> neutro
Ejemplo 3: No recomiendo este lugar, fue una experiencia horrible -> negativo
"""

In [ ]:
# Importamos las librerias necesarias
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Definimos el modelo que vamos a usar
model_path = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'

# Cargamos el tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Cargamos el modelo, recordad que este modelo ha sido preentenado para la tarea de NLI
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Esta variable almacenará la respuesta del modelo
response = []

# Para cada texto a clasificar
for premise in sentiment_sentences:

  # Concatenamos los ejemplos con el texto de entrada
  prompt = f"{few_shot_examples}\nTexto: {premise}\n->"

  # Transformamos los textos en formato de premisa e hipótesis
  premise_prompts = [prompt] * len(sentiment_labels)
  hypotheses = [label for label in sentiment_labels]

  # Tokenizamos tanto las premisas como las hipótesis
  inputs = tokenizer(premise_prompts, hypotheses, return_tensors = "pt", padding = True, truncation = True)

  # Pasamos las premisas e hipótesis tokenizadas al modelo
  outputs = model(**inputs)

  # Obtenemos los logits del modelo
  # Estos son los valores de la última capa de nuestro modelo, es decir, las probabilidades de entailment, neutral y contradiction
  logits = outputs.logits

  # Sin embargo, esto son valores en crudo
  # Vamos a usar softmax para normalizarlos, haciendo que su suma sea 1 (probabilidades excluyentes)
  probs = torch.softmax(logits, dim = 1)

  # Al estar trabajando en NLI, la salida del modelo incluye varias columnas, como entailment, neutral y contradiction
  # Nos interesa quedarnos con la columna que contenga la clase "entailment", es decir, la columna que indica
  # la probabilidad de que la premisa (texto) implique la hipotesis (cada etiqueta)
  entailment_probs = probs[:, 0]
  # No todos los modelos lo almacenan en la misma columna, así que esto puede cambiar de modelo a modelo.
  # Para saber que columna es la correcta, puedes revisar la documentación del modelo en Hugging Face

  # Vamos a realizar un paso adicional para normalizar las probabilidades
  # Este paso es necesario para que la suma del entailment de todas las etiquetas sea 1
  normalized_probs = entailment_probs / entailment_probs.sum()

  # Asociamos cada etiqueta con su probabilidad
  label_probs = {label: prob.item() for label, prob in zip(sentiment_labels, normalized_probs)}

  # Ordenamos la salida de mayor a menor según la probabilidad
  label_probs = dict(sorted(label_probs.items(), key = lambda item: item[1], reverse = True))

  # Guardamos la respuesta obtenida
  response.append({
     'text': premise,
     'scores': label_probs
  })

# Mostramos la respuesta del modelo
print(json.dumps(response, indent = 4, sort_keys = False, ensure_ascii = False))

In [ ]:
def few_shot_classification(few_shot_examples):

  # Definimos un par de textos
  sentiment_sentences = [
    "Esta película me pareció maravillosa, la volvería a ver mil veces",
    "La película me ha parecido horrible, no tiene sentido ninguno",
    "Viendo esa película me aburrí como una ostra, casi me quedo durmiendo"
  ]

  # Definimos las etiquetas a clasificar
  sentiment_labels = ["positivo", "negativo", "neutro"]

  # Esta variable almacenará la respuesta del modelo
  response = []

  # Para cada texto a clasificar
  for sentence in sentiment_sentences:

    # Creamos el prompt incluyendo los ejemplos
    prompt = f"{few_shot_examples}\nTexto: {sentence}\n->"

    # Obtenemos la respuesta del modelo
    scores = classifier(prompt, sentiment_labels)

    # Guardamos la respuesta obtenida
    response.append({
      'text': sentence,
      'scores': dict(zip(scores['labels'], scores['scores']))
    })

  return response

# Chain of Thought

In [ ]:
# Importamos las librerias necesarias
from transformers import pipeline, AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

# Definimos el modelo que vamos a usar
model_path = 'google/gemma-3-1b-it'

if torch.cuda.is_available():
  # Configuración la cuantización de 4-bit
  quantization_config = BitsAndBytesConfig(load_in_4bit=True)
else:
  quantization_config = None

# Cargamos el tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Cargamos el modelo
# Estamos cuantizando el modelo a 4-bits para que ocupe menos espacio en GPU
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    device_map="auto"
).eval() # Mantiene el modelo en el estado actual que tenía cuando se ha cargado

# Creamos el generador, definiendo la tarea de generación de texto
generator = pipeline('text-generation', model = model, tokenizer = tokenizer)

In [ ]:
# Importamos las librerias necesarias
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

# Ahora vamos a aplicar un par de ejemplos de tipo few-shot en el prompt, ademas añadimos razonamiento (chain of thought)
cot_prompt = """
Por favor, clasifica el sentimiento del siguiente texto en una de las categorías: positivo, negativo o neutro.
Primero, analiza el texto y proporciona un razonamiento breve que explique tu decisión.
Luego, da la respuesta final en este formato:

Razonamiento: [explicación]
Respuesta: [etiqueta]

Ejemplos:
Texto: Desde la semana pasada no me lo pasaba tan bien.
Razonamiento: La frase expresa que el hablante disfrutó de una experiencia reciente, lo que implica un sentimiento positivo. La comparación con un momento pasado refuerza la idea de satisfacción.
Respuesta: positivo
Texto: A tí la película te gustó, pero a mí no.
Razonamiento: La frase expresa que el hablante disfrutó de la película, pero su acompañante no. No hace ningún comentario sobre la película lo que implica un sentimiento neutro.
Respuesta: neutro.

Limitate a analizar solo el siguiente texto y a responder en el formato indicado.
Texto:"""

# Definimos un par de textos
sentiment_sentences = [
  "Esta película me pareció maravillosa",
  "La película me pareció muy mala",
  "Viendo esa película me aburrí como una ostra"
]

# Para cada texto a clasificar
for sentence in sentiment_sentences:

  # Estructuramos los mensajes de entrada en el formato requerido por Gemma
  messages = [
        {
            "role": "user",
            "content": cot_prompt + " " + sentence,
        },
  ]

  # Aplicamos un template de chat al mensaje de entrada utilizando el tokenizador.
  # Esto formatea los mensajes según el formato esperado por el modelo, añadiendo un prompt
  inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
  ).to(model.device)

  # Generamos la respuesta del modelo
  outputs = model.generate(**inputs,
                            max_new_tokens=350,
                            temperature=0.7,
                            top_k=25,
                            top_p=0.9,
                            do_sample=True
                            )

  # Decodificar la respuesta generada
  # Realizo un poco de postprocesamiento para obtener solo el nuevo texto generado
  response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("model\n")[-1]

  # Mostramos la respuesta
  print("Texto a clasificar:")
  print(sentence)
  print("Respuesta del modelo:")
  print(response)
  print()

In [ ]:
# Definimos un par de ejemplos para incluirlos en el prompt
# Los ejemplos deben estar relacionados con la tarea a resolver
cot_prompt = """
Por favor, clasifica el sentimiento del siguiente texto en una de las categorías: positivo, negativo o neutro.
Primero, analiza el texto y proporciona un razonamiento breve que explique tu decisión.
Luego, da la respuesta final en este formato:

Razonamiento: [explicación]
Respuesta: [etiqueta]

Ejemplos:
Texto: Desde la semana pasada no me lo pasaba tan bien.
Razonamiento: La frase expresa que el hablante disfrutó de una experiencia reciente, lo que implica un sentimiento positivo. La comparación con un momento pasado refuerza la idea de satisfacción.
Respuesta: positivo
Texto: A tí la película te gustó, pero a mí no.
Razonamiento: La frase expresa que el hablante disfrutó de la película, pero su acompañante no. No hace ningún comentario sobre la película lo que implica un sentimiento neutro.
Respuesta: neutro.

Limitate a analizar solo el siguiente texto y a responder en el formato indicado.
Texto:"""

prompt = f"{cot_prompt} AQUI IRÍA EL TEXTO A CLASIFICAR\nRazonamiento:\nRespuesta:"

print(prompt)